# Sample Use Case of LLM Chaining

In [2]:
import os

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# For better security, load environment variables from a .env file
from dotenv import load_dotenv




In [3]:
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

In [5]:
# Initialize Gemini model
LLM = ChatGoogleGenerativeAI(
    google_api_key=GEMINI_API_KEY, 
    model="gemini-2.5-flash"
)

E0000 00:00:1760684235.827126 5779426 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


In [6]:
# --- Prompt 1: Extract Information ---
prompt_extract = ChatPromptTemplate.from_template(
   "Extract the technical specifications from the following text:\n\n{text_input}"
)


In [7]:
# --- Prompt 2: Transform to JSON ---
prompt_transform = ChatPromptTemplate.from_template(
   "Transform the following specifications into a JSON object with 'cpu', 'memory', and 'storage' as keys:\n\n{specifications}"
)


In [8]:
# --- Build the Chain using LCEL ---
# The StrOutputParser() converts the LLM's message output to a simple string.
extraction_chain = prompt_extract | LLM | StrOutputParser()

In [11]:
# The full chain passes the output of the extraction chain into the 'specifications'
# variable for the transformation prompt.
full_chain = (
   {"specifications": extraction_chain}
   | prompt_transform
   | LLM
   | StrOutputParser()
)


In [9]:
# --- Run the Chain ---
input_text = "The new laptop model features a 3.5 GHz octa-core processor, 16GB of RAM, and a 1TB NVMe SSD."

In [10]:
extraction_chain.invoke({"text_input": input_text})

'Here are the technical specifications:\n\n*   **Processor:** 3.5 GHz octa-core\n*   **RAM:** 16GB\n*   **Storage:** 1TB NVMe SSD'

In [13]:
# Execute the chain with the input text dictionary.
final_result = full_chain.invoke({"text_input": input_text})

In [14]:
print("\n--- Final JSON Output ---")
print(final_result)


--- Final JSON Output ---
```json
{
  "cpu": "3.5 GHz octa-core",
  "memory": "16GB",
  "storage": "1TB NVMe SSD"
}
```
